# Eigenvalues, exactly — a panchi tour

An **eigenvector** of a square matrix $A$ is a non-zero vector that $A$ only stretches:

$$A\mathbf{v} = \lambda \mathbf{v}$$

The stretch factor $\lambda$ is its **eigenvalue**. This notebook walks through panchi's new exact eigen solver:

- `eigen(A)` finds eigenvalues and eigenvectors **exactly** for 2×2 and 3×3 matrices — the way you would by hand.
- Irrational answers like $\frac{1+\sqrt5}{2}$ and complex ones like $2 - i$ stay exact, thanks to the new `panchi.symbolic` layer.
- The previous numerical method lives on as `eigen_qr(A)`, for larger matrices or when no exact form exists.

> Run all cells top to bottom. Math renders as LaTeX and animations play inline.

In [ ]:
from fractions import Fraction

from IPython.display import Markdown, display

import panchi as pan
from panchi import Matrix, Vector, characteristic_polynomial, dot, eigen, eigen_qr, inverse
from panchi.algorithms import null_space
from panchi.symbolic import Polynomial, QuadraticNumber, exact_roots, sqrt
from panchi.visualizations import Animator2D, Animator3D


def plottable(vectors):
    # The plotting backends don't accept exact irrational components yet,
    # so convert to floats just for drawing.
    return [Vector([float(x) for x in v]) for v in vectors]

## 1. The by-hand method, one step at a time

Take a classic tridiagonal matrix:

$$A = \begin{bmatrix} 2 & 1 & 0 \\ 1 & 2 & 1 \\ 0 & 1 & 2 \end{bmatrix}$$

**Step 1 — the characteristic polynomial** $\det(A - \lambda I)$. For a 3×3 matrix panchi uses the textbook formula
$-\lambda^3 + \operatorname{tr}(A)\,\lambda^2 - (\text{sum of principal } 2\times2 \text{ minors})\,\lambda + \det(A)$.

In [ ]:
A = Matrix([[2, 1, 0],
            [1, 2, 1],
            [0, 1, 2]])

p = characteristic_polynomial(A)
p

**Step 2 — find a rational root.** The *rational root theorem* says any rational root $p/q$ has $p$ dividing the constant term and $q$ dividing the leading coefficient. Here the candidates are $\pm1, \pm2, \pm4$, and only one works:

In [ ]:
p.rational_roots()

**Step 3 — synthetic division.** Dividing by $(\lambda - 2)$ leaves a quadratic, with remainder $0$ confirming that $2$ is a root:

In [ ]:
quotient, remainder = p.divide_by_root(2)
display(quotient)
print("remainder:", remainder)

**Step 4 — the quadratic formula.** The discriminant isn't a perfect square, so the remaining eigenvalues are irrational — and panchi keeps them exact:

In [ ]:
for root, multiplicity in exact_roots(quotient):
    display(root)

**Step 5 — eigenvectors.** Each eigenspace is the null space of $A - \lambda I$, found by row reduction. Row reduction stays exact over $\mathbb{Q}(\sqrt2)$.

(We use `exact_matrix` so row reduction works with fractions rather than floats — mixing floats with exact square roots is refused on purpose.)

In [ ]:
lam = 2 + sqrt(2)
shifted = pan.exact_matrix(A.to_list()) - lam * pan.identity(3)
null_space(shifted)

**All at once.** `eigen(A)` runs every step above and shows the whole derivation:

In [ ]:
eigen(A)

## 2. Exact means `==`, not `≈`

Because nothing is rounded, $A\mathbf{v} = \lambda\mathbf{v}$ holds with plain equality — no tolerances:

In [ ]:
result = eigen(A)
for value, vector in result.pairs:
    print(f"λ = {value}:  A @ v == λ * v  →  {A @ vector == value * vector}")

## 3. Seeing eigenvectors

A matrix moves almost every vector off its own line. Eigenvectors are the exceptions: they stay on their line and just get stretched by $\lambda$.

Watch the transformation below. The two eigenvectors of $B$ (λ = 1 and λ = 3) stay on their lines, while the third vector — not an eigenvector — gets turned.

In [ ]:
B = Matrix([[2, 1],
            [1, 2]])
eigen(B)

In [ ]:
eigenvectors = [vector for _, vector in eigen(B).pairs]
Animator2D().animate_transform(B, vectors=eigenvectors + [Vector([1, 0])])

The **eigenspaces** are the lines those vectors span. For a symmetric matrix they are perpendicular:

In [ ]:
Animator2D().plot_span(
    eigen(B).eigenspaces,
    labels=["λ = 1", "λ = 3"],
)

## 4. Irrational eigenvalues: Fibonacci and the golden ratio

The Fibonacci matrix maps $(F_{n+1}, F_n)$ to $(F_{n+2}, F_{n+1})$:

$$\begin{bmatrix} 1 & 1 \\ 1 & 0 \end{bmatrix}$$

Its eigenvalues are the golden ratio $\varphi$ and its conjugate $\psi$ — exactly:

In [ ]:
F = Matrix([[1, 1],
            [1, 0]])
fib = eigen(F)
fib

Diagonalizing gives $F = PDP^{-1}$, so $F^{n} = PD^{n}P^{-1}$ — powering a diagonal matrix just powers its entries. Every step below is exact arithmetic in $\mathbb{Q}(\sqrt5)$, and the irrational parts cancel to give whole Fibonacci numbers:

In [ ]:
P, D = fib.diagonalization()
P_inverse = inverse(P).inverse

display(P, D, P_inverse)

In [ ]:
F10 = P @ D**10 @ P_inverse
display(F10)
print("F₁₀ =", F10[0][1])

That's **Binet's formula** in disguise: $F_n = \dfrac{\varphi^n - \psi^n}{\sqrt5}$. Powers of $\varphi$ even carry the Fibonacci numbers inside them:

In [ ]:
phi = (1 + sqrt(5)) / 2
display(phi**10)
print("φ¹⁰ - φ⁹ - φ⁸ == 0 :", phi**10 - phi**9 - phi**8 == 0)

The eigenvectors point along the golden ratio. Every Fibonacci pair $(F_{n+1}, F_n)$ gets pulled toward the $\varphi$ line:

In [ ]:
golden_line = [vector for _, vector in fib.pairs][1]
Animator2D().animate_transform(F, vectors=plottable([golden_line]) + [Vector([1, 0])])

## 5. Complex eigenvalues: rotations

A 90° rotation turns *every* vector, so it can't have a real eigenvector. Over the complex numbers it does — with eigenvalues $\pm i$:

In [ ]:
rotation = Matrix([[0, -1],
                   [1, 0]])
rot = eigen(rotation)
display(rot)
print("has complex eigenvalues:", rot.has_complex_eigenvalues)

In [ ]:
Animator2D().animate_transform(rotation, vectors=[Vector([1, 0]), Vector([1, 1]), Vector([0, 2])])

More generally, $\begin{bmatrix} a & -b \\ b & a \end{bmatrix}$ rotates *and* scales, with eigenvalues $a \pm bi$. The **modulus** $|\lambda|$ is the scale factor, and panchi computes it exactly:

In [ ]:
spiral = Matrix([[3, -4],
                 [4, 3]])
for value in eigen(spiral).eigenvalues:
    print(f"λ = {value},  |λ| = {abs(value)}")

To watch one at a comfortable size, take the same rotation with scale factor $|\lambda| = 2$: $\frac{2}{5}\begin{bmatrix} 3 & -4 \\ 4 & 3 \end{bmatrix}$. Vectors spiral outward, turning and doubling at once:

In [ ]:
gentle_spiral = Fraction(2, 5) * spiral
display(eigen(gentle_spiral))
Animator2D().animate_transform(gentle_spiral, vectors=[Vector([1, 0]), Vector([0, 1])])

## 6. When there aren't enough eigenvectors

A **shear** has eigenvalue $1$ twice (its *algebraic multiplicity* is 2), but only one line of eigenvectors (its *geometric multiplicity* is 1). Such a matrix is **defective**: its eigenvectors can't form a basis, so it can't be diagonalized.

In [ ]:
shear = Matrix([[1, 1],
                [0, 1]])
sh = eigen(shear)
display(sh)
print("algebraic:", sh.algebraic_multiplicities, " geometric:", sh.geometric_multiplicities)

In [ ]:
try:
    sh.diagonalization()
except ValueError as error:
    print("ValueError:", error)

Only the horizontal axis stays put. Every other vector gets slid sideways:

In [ ]:
Animator2D().animate_transform(shear, vectors=[Vector([1, 0]), Vector([0, 1]), Vector([-1, 1])])

## 7. A 3×3 gallery

### Symmetric matrices have orthogonal eigenvectors

The **spectral theorem** says a real symmetric matrix has real eigenvalues and perpendicular eigenvectors. We can check the second part with exact dot products:

In [ ]:
S = Matrix([[2, 0, 0],
            [0, 3, 4],
            [0, 4, 9]])
spectral = eigen(S)
display(spectral)

vectors = [vector for _, vector in spectral.pairs]
for i in range(3):
    for j in range(i + 1, 3):
        print(f"v{i + 1} · v{j + 1} = {dot(vectors[i], vectors[j])}")

In [ ]:
Animator3D().animate_transform(S, vectors=vectors)

### One real eigenvalue, one complex pair

Rational root first ($\lambda = 1$), then the quadratic formula gives a conjugate pair:

In [ ]:
eigen(Matrix([[1, 0, 0],
              [0, 1, -2],
              [0, 1, 3]]))

### A repeated eigenvalue with a whole plane of eigenvectors

Here $\lambda = 2$ is repeated, but unlike the shear its eigenspace is two-dimensional: the plane $x + y + z = 0$. Everything in that plane is simply doubled, and the perpendicular line $(1, 1, 1)$ is stretched by 5.

In [ ]:
plane = Matrix([[3, 1, 1],
                [1, 3, 1],
                [1, 1, 3]])
pl = eigen(plane)
display(pl)
print("diagonalizable:", pl.is_diagonalizable)

In [ ]:
Animator3D().plot_span(pl.eigenspaces, labels=["λ = 2 (plane)", "λ = 5 (line)"])

## 8. Where exact stops

The eigenvalues of a 3×3 matrix are exact whenever its characteristic cubic has a rational root. If it doesn't, the roots need **cube roots** — even when all three are real (the famous *casus irreducibilis*). panchi won't print an unreadable Cardano formula. It explains why instead:

In [ ]:
C = Matrix([[0, 0, -1],
            [1, 0, 3],
            [0, 1, 0]])
display(characteristic_polynomial(C))

try:
    eigen(C)
except ValueError as error:
    print("ValueError:", error)

The numerical `eigen_qr` still works, and agrees with the known closed form $2\cos\frac{2\pi k}{9}$:

In [ ]:
import math

numerical = sorted(eigen_qr(C).eigenvalues)
closed_form = sorted(2 * math.cos(2 * math.pi * k / 9) for k in (1, 2, 4))
for approx, exact in zip(numerical, closed_form):
    print(f"eigen_qr: {approx: .9f}    2cos(2πk/9): {exact: .9f}")

## 9. The symbolic toolbox

The pieces behind `eigen()` live in `panchi.symbolic`. A `QuadraticNumber` is an exact $a + b\sqrt d$ with rational $a, b$. A negative $d$ gives a complex number, so `sqrt(-1)` is $i$.

In [ ]:
display(sqrt(12), sqrt(Fraction(1, 2)), sqrt(-4), sqrt(9))

In [ ]:
z = 3 + 2 * sqrt(-1)
print("z           =", z)
print("conjugate   =", z.conjugate())
print("z · z̄       =", z * z.conjugate())
print("|z|         =", abs(z))
print("1 / z       =", 1 / z)
print("real, imag  =", z.real, ",", z.imag)

They work as `Vector` and `Matrix` entries too:

In [ ]:
Matrix([[1 + sqrt(2), Fraction(1, 2)],
        [0, 1 - sqrt(2)]])

Two deliberate guard rails: a `QuadraticNumber` holds a *single* square root, and it refuses floats rather than quietly losing exactness.

In [ ]:
for attempt in (lambda: sqrt(2) + sqrt(3), lambda: sqrt(2) * 0.5):
    try:
        attempt()
    except (ValueError, TypeError) as error:
        print(f"{type(error).__name__}: {error}")

`Polynomial` is a small exact polynomial type in its own right:

In [ ]:
cubic = Polynomial([-6, 11, -6, 1])
display(cubic)
print("rational roots:", cubic.rational_roots())
print("cubic(4) =", cubic(4))
print("(λ - 1)(λ + 1) =", Polynomial([-1, 1]) * Polynomial([1, 1]))

## 10. Now you try

Work each one out by hand first, then check with `eigen()`.

**Exercise 1.** Find the eigenvalues of $\begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}$. *Hint: the discriminant is 33.*

<details><summary>Solution</summary>

$\lambda^2 - 5\lambda - 2 = 0$, so $\lambda = \dfrac{5 \pm \sqrt{33}}{2}$.
</details>

**Exercise 2.** Find the eigenvalues and eigenvectors of $\begin{bmatrix} 1 & -1 \\ 1 & 1 \end{bmatrix}$. What does this matrix do geometrically?

<details><summary>Solution</summary>

$\lambda = 1 \pm i$, with eigenvectors $(\pm i, 1)$. Since $|1 \pm i| = \sqrt2$, it rotates by 45° and scales by $\sqrt2$.
</details>

**Exercise 3.** Find the eigenvalues of $\begin{bmatrix} 4 & 0 & 1 \\ -2 & 1 & 0 \\ -2 & 0 & 1 \end{bmatrix}$. Is it diagonalizable?

<details><summary>Solution</summary>

$\lambda = 1, 2, 3$ — three distinct eigenvalues, so yes: distinct eigenvalues always give independent eigenvectors.
</details>

In [ ]:
eigen(Matrix([[1, 2],
              [3, 4]]))

---
*Exact eigenvalues cover 1×1 through 3×3 matrices with integer or fraction entries. For anything else, use `eigen_qr`.*